# GA-Based Representation — Full Colab Notebook

Run cells in order. This notebook covers the full project goals:
- Fair comparison across models: `baseline_cnn`, `ga_cnn`, `densenet121`, `ga_densenet121`
- Core metrics: accuracy, precision, recall, F1, ECE, confusion matrix
- Interpretability: Grad-CAM outputs
- Reproducibility: config + seed control
- Methodology study: multi-seed, multi-dataset, multi-train-fraction with paired permutation tests

This notebook supports two modes:
1. **Single-run comparison** (all selected models from config)
2. **Full study mode** (`run_study`) for thesis-grade statistical analysis

In [ ]:
# ===== 1) Colab environment bootstrap =====
from pathlib import Path
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

# Optional: set your repo URL if project code is not present.
# Example: REPO_URL = "https://github.com/<owner>/<repo>.git"
REPO_URL = os.environ.get("GA_REPO_URL", "https://github.com/sarrasoussia/GA-Based-Representation-for-Image-Based-Learning-in-Medical-Imaging.git").strip()
REPO_CLONE_DIR = Path("/content/ga_project")

candidate_project_roots = [
    Path.cwd(),
    Path("/content/GA-Based Representation"),
    Path("/content/GA-Based Representation/project"),
    Path("/content/GA-Based-Representation"),
    Path("/content/GA-Based-Representation/project"),
    Path("/content/project"),
    Path("/content/ga_project"),
    Path("/content/ga_project/project"),
]

PROJECT_ROOT = None
for p in candidate_project_roots:
    if (p / "main.py").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None and REPO_URL:
    if REPO_CLONE_DIR.exists():
        shutil.rmtree(REPO_CLONE_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_CLONE_DIR)], check=True)
    if (REPO_CLONE_DIR / "main.py").exists():
        PROJECT_ROOT = REPO_CLONE_DIR
    elif (REPO_CLONE_DIR / "project" / "main.py").exists():
        PROJECT_ROOT = REPO_CLONE_DIR / "project"

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate main.py in either repo root or repo/project. Set REPO_URL (or env GA_REPO_URL) then rerun this cell."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# ===== 2) Install dependencies =====
import subprocess
from pathlib import Path

req_file = Path("requirements.txt")
if req_file.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req_file)], check=True)
else:
    fallback = ["torch", "torchvision", "numpy", "pandas", "matplotlib", "seaborn", "pyyaml", "medmnist", "tqdm", "scikit-learn"]
    subprocess.run([sys.executable, "-m", "pip", "install", *fallback], check=True)

print("Dependencies installed.")

In [ ]:
# ===== 3) Imports =====
import copy
import json
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

from data.pathmnist import denormalize_image, get_medmnist_dataloaders, labels_to_long, set_global_seed
from models.factory import build_model
from training.evaluate import evaluate_model
from training.study import run_study
from training.train import train_model
from utils.gradcam import GradCAM
from utils.visualization import plot_confusion_matrix, plot_model_comparison, plot_training_curves, save_gradcam_grid

In [ ]:
# ===== 4) Load config and device =====
CONFIG_PATH = Path("experiments/config.yaml")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

set_global_seed(cfg["seed"])

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

dataset_name = cfg.get("dataset", {}).get("name", "pathmnist")
print("Device:", device)
print("Dataset:", dataset_name)

In [ ]:
# ===== 5) Build dataloaders =====
train_fraction = float(cfg.get("dataset", {}).get("train_fraction", 1.0))

loaders = get_medmnist_dataloaders(
    dataset_name=dataset_name,
    data_dir=cfg["paths"]["data_dir"],
    batch_size=cfg["training"]["batch_size"],
    num_workers=cfg["training"]["num_workers"],
    normalize=True,
    seed=cfg["seed"],
    train_fraction=train_fraction,
)

print("Classes:", loaders.num_classes, "| Channels:", loaders.in_channels)
print("Train/Val/Test batches:", len(loaders.train), len(loaders.val), len(loaders.test))

In [ ]:
# ===== 6) Visualize sample batch =====
images, labels = next(iter(loaders.train))
images_vis = denormalize_image(images[:12]).numpy()
labels_vis = labels_to_long(labels[:12]).numpy()

fig, axes = plt.subplots(3, 4, figsize=(10, 7))
axes = axes.ravel()
for i in range(12):
    img = np.transpose(images_vis[i], (1, 2, 0))
    axes[i].imshow(img if img.shape[-1] == 3 else img.squeeze(), cmap=None if img.shape[-1] == 3 else "gray")
    axes[i].set_title(f"y={labels_vis[i]}")
    axes[i].axis("off")
plt.tight_layout()

In [ ]:
# ===== 7) Helpers =====
def _select_target_layer(model: torch.nn.Module) -> torch.nn.Module:
    if hasattr(model, "gradcam_target_layer"):
        return model.gradcam_target_layer()
    if hasattr(model, "model") and hasattr(model.model, "features"):
        return model.model.features[-1]
    if hasattr(model, "classifier") and hasattr(model.classifier, "features"):
        return model.classifier.features[-1]
    raise ValueError("Could not infer Grad-CAM target layer for model")

def run_one_model(model_name: str, cfg_local: dict, loaders, device: torch.device, out_dir: Path):
    model = build_model(
        model_name=model_name,
        in_channels=loaders.in_channels,
        num_classes=loaders.num_classes,
        include_higher_order=cfg_local["ga"].get("include_higher_order", True),
        pretrained=cfg_local.get("model", {}).get("pretrained", True),
        adapt_for_small_inputs=cfg_local.get("model", {}).get("adapt_for_small_inputs", True),
    ).to(device)

    ckpt_dir = out_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    model, hist, _ = train_model(
        model=model,
        model_name=model_name,
        train_loader=loaders.train,
        val_loader=loaders.val,
        device=device,
        save_dir=str(ckpt_dir),
        epochs=cfg_local["training"]["epochs"],
        learning_rate=cfg_local["training"]["learning_rate"],
        weight_decay=cfg_local["training"]["weight_decay"],
        early_stopping_patience=cfg_local["training"]["early_stopping_patience"],
    )

    metrics_path = out_dir / f"{model_name}_metrics.json"
    metrics = evaluate_model(model=model, loader=loaders.test, device=device, save_path=str(metrics_path))

    plot_training_curves(asdict(hist), save_path=str(out_dir / f"{model_name}_training_curves.png"), title=model_name)
    plot_confusion_matrix(
        cm=np.array(metrics["confusion_matrix"]),
        class_names=loaders.class_names,
        save_path=str(out_dir / f"{model_name}_confusion_matrix.png"),
        title=f"{model_name} - Confusion Matrix",
    )

    images, labels = next(iter(loaders.test))
    images = images[:12].to(device)
    labels_np = labels_to_long(labels[:12]).cpu().numpy()

    target_layer = _select_target_layer(model)
    cam = GradCAM(model, target_layer)
    try:
        heatmaps = cam.generate(images).detach().cpu().numpy()
        preds = torch.argmax(model(images), dim=1).detach().cpu().numpy()
    finally:
        cam.close()

    vis_images = denormalize_image(images.detach().cpu()).numpy().astype(np.float32)
    save_gradcam_grid(
        images=vis_images,
        heatmaps=heatmaps,
        preds=preds,
        labels=labels_np,
        save_path=str(out_dir / f"{model_name}_gradcam.png"),
        title=f"{model_name} Grad-CAM",
    )

    metrics_no_cm = {k: v for k, v in metrics.items() if k != "confusion_matrix"}
    return model, hist, metrics, metrics_no_cm

In [ ]:
# ===== 8) Run single-run model comparison =====
quick_cfg = copy.deepcopy(cfg)

# For full objective coverage in single-run mode, keep QUICK_RUN=False
QUICK_RUN = False
if QUICK_RUN:
    quick_cfg["training"]["epochs"] = min(5, quick_cfg["training"]["epochs"])
    quick_cfg["training"]["early_stopping_patience"] = min(3, quick_cfg["training"]["early_stopping_patience"])

# Uses experiments.run_models from config by default
models_to_run = quick_cfg.get("experiments", {}).get("run_models", ["baseline_cnn", "ga_cnn", "densenet121", "ga_densenet121"])
output_root = Path("experiments") / "results" / f"notebook_{dataset_name}"
output_root.mkdir(parents=True, exist_ok=True)

comparison = {}
for model_name in models_to_run:
    print(f"\n=== Running {model_name} ===")
    _, _, _, metrics_no_cm = run_one_model(model_name, quick_cfg, loaders, device, output_root)
    comparison[model_name] = metrics_no_cm

with open(output_root / "comparison_summary.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2)

plot_model_comparison(comparison, save_path=str(output_root / "model_comparison.png"))
comparison

In [ ]:
# ===== 9) View results =====
if comparison:
    df = pd.DataFrame(comparison).T
    display(df.sort_values(by="accuracy", ascending=False))

    fig, ax = plt.subplots(figsize=(9, 4))
    df[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_title("Notebook Run Metrics")
    plt.xticks(rotation=0)
    plt.tight_layout()

print("Saved artifacts in:", output_root.resolve())

## Optional: Full methodological study (multi-seed / multi-dataset / data efficiency)

This section executes `run_study()` using your `study` block from `experiments/config.yaml` and produces:
- `experiments/results/study/raw_results.json`
- `experiments/results/study/summary_statistics.json`

Use this for the full project goal on statistical rigor.

In [ ]:
# ===== 10) Run study mode =====
# Set to True to execute the complete study protocol from config
RUN_STUDY = False

if RUN_STUDY:
    study_cfg = cfg.get("study", {})
    print("Study settings:", study_cfg)
    run_study(cfg=cfg, device=device)
    print("Study run finished.")
else:
    print("RUN_STUDY is False. Set RUN_STUDY=True and rerun this cell for full study mode.")

In [ ]:
# ===== 11) Load and inspect study statistics =====
study_root = Path("experiments") / "results" / "study"
summary_path = study_root / "summary_statistics.json"
raw_path = study_root / "raw_results.json"

if summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        summary_stats = json.load(f)

    rows = []
    for setting, content in summary_stats.items():
        for metric, pval in content.get("paired_permutation_pvalue", {}).items():
            rows.append({
                "setting": setting,
                "metric": metric,
                "n_seeds": content.get("n_seeds", 0),
                "baseline_mean": content["baseline"][metric]["mean"],
                "ga_mean": content["ga_cnn"][metric]["mean"],
                "delta_ga_minus_baseline": content["ga_cnn"][metric]["mean"] - content["baseline"][metric]["mean"],
                "p_value": pval,
            })

    study_df = pd.DataFrame(rows).sort_values(["setting", "metric"]).reset_index(drop=True)
    display(study_df)
else:
    print("No study summary found yet. Run the previous cell with RUN_STUDY=True first.")

if raw_path.exists():
    print("Raw study results:", raw_path.resolve())
    print("Summary statistics:", summary_path.resolve())

In [ ]:
# ===== 10) Optional: compare with existing summary =====
existing_summary = Path("experiments/results/comparison_summary.json")
if existing_summary.exists():
    with open(existing_summary, "r", encoding="utf-8") as f:
        old_comp = json.load(f)
    old_df = pd.DataFrame(old_comp).T
    print("Existing summary found:")
    display(old_df)
else:
    print("No existing comparison_summary.json found at experiments/results/.")

## Optional: Pretrained models on the same study datasets

This section runs `densenet121` and `ga_densenet121` across the same datasets / fractions / seeds as your study config.
Outputs are saved in:
- `experiments/results/study_pretrained/raw_results.json`
- `experiments/results/study_pretrained/summary_statistics.json`

In [ ]:
# ===== 12) Run pretrained study =====
from utils.statistics import summarize_metric, paired_permutation_pvalue

RUN_PRETRAINED_STUDY = True  # set False to skip
PRETRAINED_MODELS = ["densenet121", "ga_densenet121"]

if RUN_PRETRAINED_STUDY:
    study_cfg = cfg.get("study", {})
    datasets = study_cfg.get("datasets", [cfg.get("dataset", {}).get("name", "pathmnist")])
    seeds = study_cfg.get("seeds", [cfg.get("seed", 42)])
    train_fractions = study_cfg.get("train_fractions", [1.0])

    pretrained_root = Path("experiments") / "results" / "study_pretrained"
    pretrained_root.mkdir(parents=True, exist_ok=True)

    pretrained_raw = []

    for dataset_name_local in datasets:
        for train_fraction_local in train_fractions:
            for seed_local in seeds:
                set_global_seed(seed_local)
                loaders_local = get_medmnist_dataloaders(
                    dataset_name=dataset_name_local,
                    data_dir=cfg["paths"]["data_dir"],
                    batch_size=cfg["training"]["batch_size"],
                    num_workers=cfg["training"]["num_workers"],
                    normalize=True,
                    seed=seed_local,
                    train_fraction=float(train_fraction_local),
                )

                run_dir = pretrained_root / dataset_name_local / f"fraction_{float(train_fraction_local):.2f}" / f"seed_{seed_local}"
                run_dir.mkdir(parents=True, exist_ok=True)

                print(f"[Pretrained Study] dataset={dataset_name_local} fraction={float(train_fraction_local):.2f} seed={seed_local}")
                row = {
                    "dataset": dataset_name_local,
                    "train_fraction": float(train_fraction_local),
                    "seed": int(seed_local),
                }

                for model_name_local in PRETRAINED_MODELS:
                    _, _, metrics_local, _ = run_one_model(
                        model_name=model_name_local,
                        cfg_local=cfg,
                        loaders=loaders_local,
                        device=device,
                        out_dir=run_dir,
                    )
                    row[model_name_local] = {
                        "accuracy": metrics_local["accuracy"],
                        "precision_macro": metrics_local["precision_macro"],
                        "recall_macro": metrics_local["recall_macro"],
                        "f1_macro": metrics_local["f1_macro"],
                        "ece": metrics_local["ece"],
                    }

                pretrained_raw.append(row)

    with open(pretrained_root / "raw_results.json", "w", encoding="utf-8") as f:
        json.dump(pretrained_raw, f, indent=2)

    metrics_list = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "ece"]
    summary = {}

    for dataset_name_local in datasets:
        for train_fraction_local in train_fractions:
            key = f"{dataset_name_local}|fraction={float(train_fraction_local):.2f}"
            group = [
                r for r in pretrained_raw
                if r["dataset"] == dataset_name_local and abs(r["train_fraction"] - float(train_fraction_local)) < 1e-9
            ]

            densenet_vals = {m: [r["densenet121"][m] for r in group] for m in metrics_list}
            ga_dense_vals = {m: [r["ga_densenet121"][m] for r in group] for m in metrics_list}

            pvals = {
                m: paired_permutation_pvalue(densenet_vals[m], ga_dense_vals[m])
                for m in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
            }

            summary[key] = {
                "n_seeds": len(group),
                "densenet121": {m: summarize_metric(densenet_vals[m]) for m in metrics_list},
                "ga_densenet121": {m: summarize_metric(ga_dense_vals[m]) for m in metrics_list},
                "paired_permutation_pvalue": pvals,
            }

    with open(pretrained_root / "summary_statistics.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("Pretrained study complete.")
    print("Saved:", (pretrained_root / "raw_results.json").resolve())
    print("Saved:", (pretrained_root / "summary_statistics.json").resolve())
else:
    print("RUN_PRETRAINED_STUDY is False. Set it to True to run pretrained study.")

## Notes
- For complete project goals, run both sections: single-run comparison + study mode.
- To run pretrained models on the same datasets/fractions/seeds, run cell 16 (`RUN_PRETRAINED_STUDY=True`).
- Single-run outputs are saved in `experiments/results/notebook_<dataset>/`.
- CNN/GA-CNN study outputs are saved in `experiments/results/study/`.
- Pretrained study outputs are saved in `experiments/results/study_pretrained/`.